# Notebook 09: Final Evaluation and Analysis

Comprehensive comparison of all model variants with statistical significance testing, qualitative analysis, and publication-ready figures.

**Models evaluated:**
1. `text-embedding-3-small` (baseline, frozen)
2. `text-embedding-3-large` (baseline, frozen)
3. Bi-encoder + random negatives (Variant 2)
4. Bi-encoder + hard negatives (Variant 3)
5. Hierarchical contrastive loss (Variant 4)
6. Tri-modal alignment (Variant 5)

**Prerequisite:** Run notebooks 01-08 first.

In [ ]:
import os, sys, json, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_random_negatives, build_category_sibling_negatives, build_dfsdt_negatives
from models.embeddings import format_api_string, format_api_code
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import recall_at_k, mean_reciprocal_rank, evaluate_batch

In [ ]:
corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)
evals = load_eval_examples(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json')

with open(PROJECT_DIR / 'api_names.json') as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}

with open(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json') as f:
    raw_evals = json.load(f)

api_strings = [format_api_string(a) for a in corpus]
code_strings = [format_api_code(a) for a in corpus]
queries = [e['user_query'] for e in evals]

print(f'Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples')

## Encode All Models

Load fine-tuned checkpoints and compute corpus/query embeddings for each variant.

In [ ]:
from sentence_transformers import SentenceTransformer

model_configs = [
    ('v2_random', 'checkpoints/v2_random', False),
    ('v3_hard', 'checkpoints/v3_hard', False),
    ('v4_hierarchical', 'checkpoints/v4_hierarchical', False),
    ('v5_trimodal', 'checkpoints/v5_trimodal', True),
]

embeddings = {}
for label, path, trimodal in model_configs:
    model_path = PROJECT_DIR / path
    if not model_path.exists():
        print(f'Skipping {label}: checkpoint not found')
        continue
    print(f'Encoding with {label}...')
    model = SentenceTransformer(str(model_path))
    corpus_embs = model.encode(api_strings, show_progress_bar=True, batch_size=256)
    if trimodal:
        code_embs = model.encode(code_strings, show_progress_bar=True, batch_size=256)
        corpus_embs = (corpus_embs + code_embs) / 2
    embeddings[label] = {
        'corpus': corpus_embs,
        'queries': model.encode(queries, batch_size=256),
    }

print(f'Loaded {len(embeddings)} fine-tuned models')

## 1. Comprehensive Comparison Table

Full-corpus retrieval and restricted 100-candidate pool evaluation across all models and negative conditions.

In [ ]:
def per_example_scores(query_embs, corpus_embs, ks=[1, 5, 10]):
    """Return per-example recall and MRR scores for bootstrap analysis."""
    index = build_faiss_index(corpus_embs)
    scores = []
    for i, ex in enumerate(evals):
        top_k = retrieve_top_k(query_embs[i], index, k=max(ks))
        gt = [name_to_idx[n] for n in ex['ground_truth_apis'] if n in name_to_idx]
        row = {}
        for k in ks:
            row[f'r@{k}'] = recall_at_k(top_k, gt, k)
        row['mrr'] = mean_reciprocal_rank(top_k, gt)
        row['top_k'] = top_k
        row['gt'] = gt
        scores.append(row)
    return scores

def per_example_restricted(query_embs, corpus_embs, neg_type, n_neg=99, ks=[1, 5, 10]):
    """Return per-example scores for restricted pool evaluation."""
    scores = []
    for i, ex in enumerate(evals):
        raw_ex = raw_evals[ex['raw_idx']]
        gt_names = ex['ground_truth_apis']
        gt_indices = [name_to_idx[n] for n in gt_names if n in name_to_idx]
        if not gt_indices:
            continue
        if neg_type == 'random':
            negs = build_random_negatives(corpus, gt_names, n=n_neg)
        elif neg_type == 'sibling':
            negs = build_category_sibling_negatives(corpus, gt_names, lookup, n=n_neg)
        else:
            negs = build_dfsdt_negatives(raw_ex, corpus, gt_names, lookup, n=n_neg)
        cand_indices = [name_to_idx[n] for n in gt_names + [a['action_name'] for a in negs] if n in name_to_idx]
        if len(cand_indices) < 2:
            continue
        local_index = build_faiss_index(corpus_embs[cand_indices].astype(np.float32))
        g2l = {g: j for j, g in enumerate(cand_indices)}
        local_gt = [g2l[g] for g in gt_indices if g in g2l]
        if not local_gt:
            continue
        top_k = retrieve_top_k(query_embs[i], local_index, k=min(max(ks), len(cand_indices)))
        row = {'example_idx': i}
        for k in ks:
            row[f'r@{k}'] = recall_at_k(top_k, local_gt, k)
        row['mrr'] = mean_reciprocal_rank(top_k, local_gt)
        scores.append(row)
    return scores

print('Scoring functions defined.')

In [ ]:
# Load baseline embeddings for per-example scoring
from models.embeddings import get_embeddings

baseline_corpus = np.load(PROJECT_DIR / 'corpus_embeddings.npy')
baseline_queries = get_embeddings(queries)

# Full-corpus per-example scores for all models
all_full_scores = {}
all_full_scores['baseline'] = per_example_scores(baseline_queries, baseline_corpus)

for label, embs in embeddings.items():
    print(f'Full-corpus: {label}...')
    all_full_scores[label] = per_example_scores(embs['queries'], embs['corpus'])

# Aggregate results table
full_results = {}
for label, scores in all_full_scores.items():
    full_results[label] = {
        f'recall@{k}': np.mean([s[f'r@{k}'] for s in scores])
        for k in [1, 5, 10]
    }
    full_results[label]['mrr'] = np.mean([s['mrr'] for s in scores])

print(f"\n{'Model':<30} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 62)
for label, r in full_results.items():
    print(f"{label:<30} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

In [ ]:
# Hard-negative ablation per-example scores
all_hn_scores = {}

# Baseline
all_hn_scores['baseline'] = {}
for neg_type in ['random', 'sibling', 'dfsdt']:
    print(f'baseline / {neg_type}...')
    all_hn_scores['baseline'][neg_type] = per_example_restricted(baseline_queries, baseline_corpus, neg_type)

# Fine-tuned models
for label, embs in embeddings.items():
    all_hn_scores[label] = {}
    for neg_type in ['random', 'sibling', 'dfsdt']:
        print(f'{label} / {neg_type}...')
        all_hn_scores[label][neg_type] = per_example_restricted(embs['queries'], embs['corpus'], neg_type)

# Aggregate
hn_results = {}
for model_label, conditions in all_hn_scores.items():
    hn_results[model_label] = {}
    for cond, scores in conditions.items():
        hn_results[model_label][cond] = {
            f'recall@{k}': np.mean([s[f'r@{k}'] for s in scores])
            for k in [1, 5, 10]
        }
        hn_results[model_label][cond]['mrr'] = np.mean([s['mrr'] for s in scores])

print(f"\n{'Model / Condition':<45} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 77)
for model_label, conditions in hn_results.items():
    for cond, r in conditions.items():
        tag = f'{model_label} / {cond}'
        print(f"{tag:<45} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

## 2. Statistical Significance

Bootstrap confidence intervals (95%) on the difference between each fine-tuned model and the baseline. A difference is significant if the CI excludes zero.

In [ ]:
def bootstrap_ci(scores_a, scores_b, metric, n_boot=10000, alpha=0.05):
    """Bootstrap 95% CI for mean(scores_b[metric]) - mean(scores_a[metric])."""
    a = np.array([s[metric] for s in scores_a])
    b = np.array([s[metric] for s in scores_b])
    n = min(len(a), len(b))
    a, b = a[:n], b[:n]
    diffs = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs.append(b[idx].mean() - a[idx].mean())
    diffs = np.sort(diffs)
    lo = diffs[int(alpha / 2 * n_boot)]
    hi = diffs[int((1 - alpha / 2) * n_boot)]
    mean_diff = np.mean(diffs)
    sig = 'Yes' if (lo > 0 or hi < 0) else 'No'
    return mean_diff, lo, hi, sig

# Full-corpus significance tests
baseline_scores = all_full_scores['baseline']

print(f"{'Model vs baseline':<25} {'Metric':<8} {'Diff':>8} {'95% CI':>20} {'Sig?':>6}")
print('-' * 70)
for label in embeddings:
    model_scores = all_full_scores[label]
    for metric in ['r@5', 'mrr']:
        diff, lo, hi, sig = bootstrap_ci(baseline_scores, model_scores, metric)
        print(f"{label:<25} {metric:<8} {diff:>+8.4f} [{lo:>+8.4f}, {hi:>+8.4f}] {sig:>6}")

In [ ]:
# Hard-negative significance: compare each model on DFSDT condition vs baseline
print(f"{'Model vs baseline (DFSDT)':<25} {'Metric':<8} {'Diff':>8} {'95% CI':>20} {'Sig?':>6}")
print('-' * 70)
baseline_dfsdt = all_hn_scores['baseline']['dfsdt']
for label in embeddings:
    model_dfsdt = all_hn_scores[label]['dfsdt']
    for metric in ['r@1', 'r@5', 'mrr']:
        diff, lo, hi, sig = bootstrap_ci(baseline_dfsdt, model_dfsdt, metric)
        print(f"{label:<25} {metric:<8} {diff:>+8.4f} [{lo:>+8.4f}, {hi:>+8.4f}] {sig:>6}")

## 3. Ablation: Negative Mining Strategy

Compare how each model handles different negative difficulties. The gap between random and DFSDT performance reveals robustness to hard negatives.

In [ ]:
# Performance degradation from random to DFSDT negatives (R@5)
print(f"{'Model':<25} {'R@5 random':>12} {'R@5 dfsdt':>12} {'Drop':>8} {'Drop %':>8}")
print('-' * 65)
for label in ['baseline'] + list(embeddings.keys()):
    r_rand = hn_results[label]['random']['recall@5']
    r_dfsdt = hn_results[label]['dfsdt']['recall@5']
    drop = r_rand - r_dfsdt
    drop_pct = drop / r_rand * 100 if r_rand > 0 else 0
    print(f"{label:<25} {r_rand:>12.4f} {r_dfsdt:>12.4f} {drop:>8.4f} {drop_pct:>7.1f}%")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))
model_labels = ['baseline'] + list(embeddings.keys())
x = np.arange(len(model_labels))
width = 0.25

for i, neg_type in enumerate(['random', 'sibling', 'dfsdt']):
    vals = [hn_results[m][neg_type]['recall@5'] for m in model_labels]
    ax.bar(x + i * width, vals, width, label=neg_type)

ax.set_ylabel('Recall@5')
ax.set_title('Recall@5 by Negative Difficulty')
ax.set_xticks(x + width)
ax.set_xticklabels(model_labels, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'ablation_negatives.png', dpi=150)
plt.show()

## 4. Qualitative Analysis

### 4a. Examples where fine-tuning fixed baseline failures

Find queries where the baseline missed at R@5 but the best fine-tuned model succeeds.

In [ ]:
# Pick the best fine-tuned model by full-corpus R@5
best_model = max(embeddings.keys(), key=lambda m: full_results[m]['recall@5'])
print(f'Best fine-tuned model: {best_model}\n')

baseline_sc = all_full_scores['baseline']
best_sc = all_full_scores[best_model]

# Cases where baseline R@5=0 but fine-tuned R@5>0
fixed = []
for i in range(len(evals)):
    if baseline_sc[i]['r@5'] == 0 and best_sc[i]['r@5'] > 0:
        fixed.append(i)

print(f'Fixed by fine-tuning: {len(fixed)} / {len(evals)} examples\n')
for rank, i in enumerate(fixed[:5], 1):
    ex = evals[i]
    print(f'--- Fixed #{rank} ---')
    print(f'Query: {ex["user_query"][:200]}')
    print(f'Expected: {ex["ground_truth_apis"][:3]}')
    baseline_top5 = [api_names[j] for j in baseline_sc[i]['top_k'][:5]]
    best_top5 = [api_names[j] for j in best_sc[i]['top_k'][:5]]
    print(f'Baseline top-5: {baseline_top5}')
    print(f'{best_model} top-5: {best_top5}\n')

### 4b. Remaining failure cases

Queries where even the best fine-tuned model fails at R@10.

In [ ]:
# Cases where all models fail at R@10
still_fail = []
for i in range(len(evals)):
    all_fail = all(all_full_scores[m][i]['r@10'] == 0 for m in all_full_scores)
    if all_fail:
        still_fail.append(i)

print(f'Still failing at R@10 across all models: {len(still_fail)} / {len(evals)}\n')
for rank, i in enumerate(still_fail[:5], 1):
    ex = evals[i]
    gt_cats = {lookup[n]['category'] for n in ex['ground_truth_apis'] if n in lookup}
    print(f'--- Failure #{rank} ---')
    print(f'Query: {ex["user_query"][:200]}')
    print(f'Expected: {ex["ground_truth_apis"][:3]}')
    print(f'Categories: {gt_cats}')
    best_top5 = [api_names[j] for j in best_sc[i]['top_k'][:5]]
    print(f'{best_model} top-5: {best_top5}\n')

### 4c. t-SNE Visualization

Visualize the embedding space before and after fine-tuning, colored by API category.

In [ ]:
from sklearn.manifold import TSNE

# Sample 2000 APIs for t-SNE (full corpus is too large)
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(corpus), size=min(2000, len(corpus)), replace=False)
sample_cats = [corpus[i]['category'] for i in sample_idx]

# Assign colors to top-10 categories, rest as "Other"
from collections import Counter
cat_counts = Counter(sample_cats)
top_cats = [c for c, _ in cat_counts.most_common(10)]
cat_labels = [c if c in top_cats else 'Other' for c in sample_cats]
unique_labels = top_cats + ['Other']
color_map = {c: i for i, c in enumerate(unique_labels)}
colors = [color_map[c] for c in cat_labels]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Baseline embeddings
tsne_base = TSNE(n_components=2, perplexity=30, random_state=42)
proj_base = tsne_base.fit_transform(baseline_corpus[sample_idx])
sc1 = axes[0].scatter(proj_base[:, 0], proj_base[:, 1], c=colors, cmap='tab20', s=3, alpha=0.6)
axes[0].set_title('Baseline (text-embedding-3-small)')
axes[0].set_xticks([])
axes[0].set_yticks([])

# Best fine-tuned embeddings
best_corpus = embeddings[best_model]['corpus']
tsne_ft = TSNE(n_components=2, perplexity=30, random_state=42)
proj_ft = tsne_ft.fit_transform(best_corpus[sample_idx].astype(np.float32))
sc2 = axes[1].scatter(proj_ft[:, 0], proj_ft[:, 1], c=colors, cmap='tab20', s=3, alpha=0.6)
axes[1].set_title(f'Fine-tuned ({best_model})')
axes[1].set_xticks([])
axes[1].set_yticks([])

# Shared legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.tab20(color_map[c] / len(unique_labels)), markersize=6, label=c[:25]) for c in unique_labels]
fig.legend(handles=handles, loc='center right', bbox_to_anchor=(1.15, 0.5), fontsize=8)
plt.suptitle('API Embedding Space (t-SNE, colored by category)', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'tsne_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Publication Figures

### 5a. Main results bar chart (full-corpus R@1, R@5, R@10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
model_labels = ['baseline'] + list(embeddings.keys())
display_names = {
    'baseline': 'Baseline\n(emb-3-small)',
    'v2_random': 'V2\nRandom Neg',
    'v3_hard': 'V3\nHard Neg',
    'v4_hierarchical': 'V4\nHierarchical',
    'v5_trimodal': 'V5\nTri-Modal',
}
x = np.arange(len(model_labels))
width = 0.22

for i, metric in enumerate(['recall@1', 'recall@5', 'recall@10']):
    vals = [full_results[m][metric] for m in model_labels]
    bars = ax.bar(x + i * width, vals, width, label=metric.replace('recall@', 'R@'))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_ylabel('Score')
ax.set_title('Full-Corpus Retrieval Performance')
ax.set_xticks(x + width)
ax.set_xticklabels([display_names.get(m, m) for m in model_labels], fontsize=9)
ax.legend()
ax.set_ylim(0, max(full_results[m]['recall@10'] for m in model_labels) * 1.15)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'fig_full_corpus.png', dpi=200)
plt.show()

### 5b. Per-category improvement heatmap

Show R@5 improvement over baseline for each model across the hardest categories.

In [ ]:
# Per-category R@5 for each model
cat_model_scores = {}
for model_label in ['baseline'] + list(embeddings.keys()):
    scores = all_full_scores[model_label]
    cat_scores = defaultdict(list)
    for i, ex in enumerate(evals):
        cats = {lookup[n]['category'] for n in ex['ground_truth_apis'] if n in lookup}
        for cat in cats:
            cat_scores[cat].append(scores[i]['r@5'])
    cat_model_scores[model_label] = {cat: np.mean(v) for cat, v in cat_scores.items() if len(v) >= 5}

# Find hardest categories (lowest baseline R@5)
baseline_cat = cat_model_scores['baseline']
hard_cats = sorted(baseline_cat.items(), key=lambda x: x[1])[:15]
hard_cat_names = [c for c, _ in hard_cats]

# Build heatmap matrix: improvement over baseline
ft_models = list(embeddings.keys())
matrix = np.zeros((len(hard_cat_names), len(ft_models)))
for j, m in enumerate(ft_models):
    for i, cat in enumerate(hard_cat_names):
        bl = baseline_cat.get(cat, 0)
        ft = cat_model_scores[m].get(cat, 0)
        matrix[i, j] = ft - bl

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=-0.2, vmax=0.2)
ax.set_xticks(range(len(ft_models)))
ax.set_xticklabels([display_names.get(m, m).replace('\n', ' ') for m in ft_models], rotation=30, ha='right')
ax.set_yticks(range(len(hard_cat_names)))
ax.set_yticklabels([c[:30] for c in hard_cat_names], fontsize=8)
for i in range(len(hard_cat_names)):
    for j in range(len(ft_models)):
        ax.text(j, i, f'{matrix[i, j]:+.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, label='R@5 improvement over baseline')
ax.set_title('Per-Category R@5 Improvement (hardest categories)')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'fig_category_heatmap.png', dpi=200)
plt.show()

## 6. Save All Results

In [ ]:
# Compute bootstrap CIs for the final results JSON
significance = {}
for label in embeddings:
    significance[label] = {}
    for metric in ['r@1', 'r@5', 'r@10', 'mrr']:
        diff, lo, hi, sig = bootstrap_ci(baseline_scores, all_full_scores[label], metric)
        significance[label][metric] = {'diff': diff, 'ci_lo': lo, 'ci_hi': hi, 'significant': sig == 'Yes'}

final_results = {
    'full_corpus': full_results,
    'hard_negative_ablation': hn_results,
    'significance_vs_baseline': significance,
    'qualitative': {
        'n_fixed_by_finetuning': len(fixed),
        'n_still_failing': len(still_fail),
        'best_model': best_model,
    },
}
with open(PROJECT_DIR / 'results_final.json', 'w') as f:
    json.dump(final_results, f, indent=2)
print('Saved results_final.json')

# Summary
print(f'\nBest model: {best_model}')
print(f'Full-corpus R@5: {full_results["baseline"]["recall@5"]:.4f} (baseline) -> {full_results[best_model]["recall@5"]:.4f} ({best_model})')
print(f'DFSDT R@5: {hn_results["baseline"]["dfsdt"]["recall@5"]:.4f} (baseline) -> {hn_results[best_model]["dfsdt"]["recall@5"]:.4f} ({best_model})')
print(f'Fixed {len(fixed)} baseline failures, {len(still_fail)} remain intractable')